<a href="https://colab.research.google.com/github/RavisriWelabada/NIM-Optimization-Business-Analytics-Seminar-Project-Group-No-14/blob/main/NIM_Optimization_Business_Analytics_Seminar_Group_No_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Import Libraries & Load Data

SQLite Database Created

In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv ('https://raw.githubusercontent.com/RavisriWelabada/NIM-Optimization-Business-Analytics-Seminar-Project-Group-No-14/main/Synthetic_NIM_Optimization_Data.csv')

# Create SQLite database
conn = sqlite3.connect("nim_database.db")

# Save table
df.to_sql("nim_data", conn, if_exists="replace", index=False)

conn.close()

In [ ]:
conn = sqlite3.connect("nim_database.db")

df_check = pd.read_sql("SELECT * FROM nim_data LIMIT 5", conn)
print(df_check)

conn.close()

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML Models
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Load dataset
# Corrected URL to point to the raw CSV file on GitHub
df = pd.read_csv ('https://raw.githubusercontent.com/RavisriWelabada/NIM-Optimization-Business-Analytics-Seminar-Project-Group-No-14/main/Synthetic_NIM_Optimization_Data.csv')

# Create SQLite database
conn = sqlite3.connect("nim_database.db")

# Save table
df.to_sql("nim_data", conn, if_exists="replace", index=False)

conn.close()

# Preview
print(df.head())
print(df.info())

2. Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics
print(df.describe())

# Check missing values
print(df.isnull().sum())

# Distribution plots
sns.histplot(df['Current Balance'], kde=True)
plt.title('Current Balance Distribution')
plt.show()

sns.countplot(x='ACCOUNT_TYPE', data=df)
plt.title('Account Type Distribution')
plt.show()

3. CASA Ratio Calculation

In [ ]:
# CASA = Savings + Current
casa_balance = df[df['ACCOUNT_TYPE'].isin(['SAVINGS', 'CURRENT'])]['Current Balance'].sum()
total_deposits = df[df['ACCOUNT_TYPE'].isin(['SAVINGS', 'CURRENT', 'FD'])]['Current Balance'].sum()

casa_ratio = casa_balance / total_deposits
print("CASA Ratio:", casa_ratio)

4. NPL Ratio

In [ ]:
loan_df = df[df['ACCOUNT_TYPE'] == 'LOAN']

npl_ratio = (loan_df['NPL TAG'] == 'Y').sum() / len(loan_df)
print("NPL Ratio:", npl_ratio)

5. NIM Calculation

In [ ]:
total_interest_earned = (df[df['ACCOUNT_TYPE'] == 'LOAN']['CURRENT INT RATE'] * df[df['ACCOUNT_TYPE'] == 'LOAN']['Current Balance']).sum()
total_interest_paid = (df[df['ACCOUNT_TYPE'].isin(['FD', 'SAVINGS', 'CURRENT'])]['CURRENT INT RATE'] * df[df['ACCOUNT_TYPE'].isin(['FD', 'SAVINGS', 'CURRENT'])]['Current Balance']).sum()

# Earning assets are primarily loans
total_earning_assets = df[df['ACCOUNT_TYPE'] == 'LOAN']['Current Balance'].sum()

# Avoid division by zero
if total_earning_assets == 0:
    nim = 0
else:
    nim = (total_interest_earned - total_interest_paid) / total_earning_assets

print("Estimated NIM:", nim)

6. K-Means Customer Segmentation

In [ ]:
import pandas as pd
from datetime import datetime
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure 'Tenure_Days' and encoded columns are present for segmentation
# This block ensures dependencies are met if data preparation was not executed or df was overwritten.

# Convert 'DATE OPEN' to datetime if not already, then calculate 'Tenure_Days'
if not pd.api.types.is_datetime64_any_dtype(df['DATE OPEN']):
    df['DATE OPEN'] = pd.to_datetime(df['DATE OPEN'])
current_date = pd.to_datetime('2025-01-01') # Use a fixed current date for consistent tenure calculation
df['Tenure_Days'] = (current_date - df['DATE OPEN']).dt.days

# Encode categorical variables if not already done
le = LabelEncoder()
if 'SEGMENT' in df.columns and 'Segment_Encoded' not in df.columns:
    df['Segment_Encoded'] = le.fit_transform(df['SEGMENT'])
if 'ACCOUNT_TYPE' in df.columns and 'Type_Encoded' not in df.columns:
    df['Type_Encoded'] = le.fit_transform(df['ACCOUNT_TYPE'])
if 'NPL TAG' in df.columns and 'NPL_Encoded' not in df.columns:
    df['NPL_Encoded'] = le.fit_transform(df['NPL TAG'])


# Select features using available columns from the DataFrame
features = df[['Current Balance', 'Tenure_Days', 'CURRENT INT RATE']]

# Scale data
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# KMeans
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10) # Added n_init to suppress future warning
df['Cluster'] = kmeans.fit_predict(scaled_features)

# Visualization - updated to use available columns
sns.scatterplot(x=df['Current Balance'], y=df['Tenure_Days'], hue=df['Cluster'])
plt.title("Customer Segmentation")
plt.xlabel("Current Balance")
plt.ylabel("Tenure (Days)")
plt.show()

# Cluster summary
print(df.groupby('Cluster')[['Current Balance', 'Tenure_Days', 'CURRENT INT RATE']].mean())

7. Random Forest – NPL Prediction (Risk-Based Pricing)

In [ ]:
# Prepare data
# Use the correct column name 'ACCOUNT_TYPE' and filter for 'LOAN'
loan_df = df[df['ACCOUNT_TYPE'] == 'LOAN'].copy()

# Select relevant features that exist in the DataFrame
# Using 'Current Balance', 'CURRENT INT RATE', 'Tenure_Days', and 'Segment_Encoded' as features
X = loan_df[['Current Balance', 'CURRENT INT RATE', 'Tenure_Days', 'Segment_Encoded']]

# Use the NPL_Encoded column for the target variable
y = loan_df['NPL_Encoded']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)

# Evaluation
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

8. LSTM Model – Balance Forecasting

In [ ]:
# Prepare data
# Correcting 'Account_Type' to 'ACCOUNT_TYPE' and using existing columns for features and target
loan_df = df[df['ACCOUNT_TYPE'] == 'LOAN'].copy()

X = loan_df[['Current Balance', 'CURRENT INT RATE', 'Tenure_Days', 'Segment_Encoded']]
y = loan_df['NPL_Encoded']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Model
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)

# Evaluation
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

9. Monte Carlo Simulation for NIM

In [ ]:
simulations = 1000
nim_results = []

# Separate loan and deposit dataframes based on 'ACCOUNT_TYPE'
loan_df_for_sim = df[df['ACCOUNT_TYPE'] == 'LOAN'].copy()
deposit_df_for_sim = df[df['ACCOUNT_TYPE'].isin(['FD', 'SAVINGS', 'CURRENT'])].copy()

# Ensure there's data to avoid division by zero or errors on empty DFs
if loan_df_for_sim.empty or deposit_df_for_sim.empty:
    print("Not enough data for simulation (empty loan or deposit accounts).")
else:
    # Get the mean rates for loans and deposits from the actual data
    mean_loan_rate = loan_df_for_sim['CURRENT INT RATE'].mean()
    mean_deposit_rate = deposit_df_for_sim['CURRENT INT RATE'].mean()

    # Calculate total earning assets for the denominator, which are primarily loans
    total_earning_assets = loan_df_for_sim['Current Balance'].sum()

    if total_earning_assets == 0:
        print("Total earning assets is zero, cannot calculate NIM.")
    else:
        for i in range(simulations):
            # Simulate loan interest rates
            simulated_loan_rates = np.random.normal(mean_loan_rate, 0.01, len(loan_df_for_sim))
            # Ensure rates are positive
            simulated_loan_rates[simulated_loan_rates < 0] = 0

            # Simulate deposit interest rates
            simulated_deposit_rates = np.random.normal(mean_deposit_rate, 0.005, len(deposit_df_for_sim))
            simulated_deposit_rates[simulated_deposit_rates < 0] = 0 # Ensure rates are positive

            # Calculate total interest earned from loans
            loan_income = (simulated_loan_rates * loan_df_for_sim['Current Balance']).sum()
            # Calculate total interest paid on deposits
            deposit_cost = (simulated_deposit_rates * deposit_df_for_sim['Current Balance']).sum()

            nim_sim = (loan_income - deposit_cost) / total_earning_assets
            nim_results.append(nim_sim)

        # Plot results
        plt.hist(nim_results, bins=50)
        plt.title("Monte Carlo Simulation of NIM")
        plt.xlabel("NIM")
        plt.ylabel("Frequency")
        plt.show()

        print("Average Simulated NIM:", np.mean(nim_results))

10. Decision Support Output (Final Insight)

In [ ]:
print("=== FINAL INSIGHTS ===")
print("CASA Ratio:", casa_ratio)
print("NPL Ratio:", npl_ratio)
print("Current NIM:", nim)
print("Simulated NIM (Avg):", np.mean(nim_results))